In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from dotenv import load_dotenv

load_dotenv()

from myapp import create_app

app = create_app()
app.app_context().push()

In [2]:
from myapp.extensions import db
from sqlalchemy import select

# service
from myapp.features.auth.services import authenticate

# repository
from myapp.features.auth.repository import user_repository, UserRepository

# models
from myapp.features.auth.models import User

# utils
from myapp.features.auth.utils import *

from werkzeug.security import check_password_hash

use cases

In [3]:
from myapp.features.auth import usecases

In [6]:
repo = UserRepository()
repo.get_all()
user = repo.first()

In [8]:
user.username

'Desaku'

In [ ]:
user_repository.list_users()

In [ ]:
# ganti password

uc = usecases.GantiPasswordAdmin()
uc.execute(1, "1234@Admin", "123@Admin")

In [ ]:
from myapp.features.auth.repository import user_repository

In [ ]:
username = "Desaku"
password = "123@Admin"

In [ ]:
db.session.get(User, 1)

In [ ]:
user = user_repository.get_user_by_id(1)
user
hashed = user.password
hashed

In [ ]:
import bcrypt

password = "123@Admin"
hashed = b"$2y$10$l.833iKxH0adHbPArVETiexCZplxqIBZw4on29xopnLMVro17Bm.a"

bcrypt.checkpw(password.encode(), hashed)

In [ ]:
import bcrypt

password = "123@Admin"
hashed = "$2y$10$l.833iKxH0adHbPArVETiexCZplxqIBZw4on29xopnLMVro17Bm.a"

# print(bcrypt.checkpw(password, hashed))

In [ ]:
from myapp.features.auth.utils import hash_password

# update password admin 
password_baru = "123@Admin"
user.password = hash_password(password_baru)

user_repository.update_user(user)

In [ ]:
import bcrypt

def check_password(password: str, hashed: str):
    return bcrypt.checkpw(
        password.encode(),
        hashed.encode()
    )
    
    
password = "123@Admin"
user = user_repository.get_user_by_id(1)
user
hashed = user.password

check_password(password, hashed)

In [ ]:
def check_password(password: str, hashed: str) -> bool:
    """
    Membandingkan password plain dengan hash
    """
    return bcrypt.checkpw(
        password.encode('utf-8'),
        hashed.encode('utf-8')
    )

In [ ]:
check_password(password, hashed)

In [ ]:
print(user.username)
print(user.password)
print(len(user.password))

In [ ]:
import re
import hashlib
import bcrypt


def validate_password(user, request: dict):
    """
    user:
        object user dari database
        contoh:
        user.password = "$2b$12$...."

    request:
        {
            "pass_lama": "...",
            "pass_baru": "...",
            "pass_baru1": "..."
        }
    """

    pass_lama = request.get("pass_lama")
    pass_baru = request.get("pass_baru")
    pass_baru1 = request.get("pass_baru1")

    # cek apakah password lama masih MD5
    pw_masih_md5 = (
        len(user.password) == 32
        and "$" not in user.password
    )

    # 1. Password kosong
    if not pass_lama or not pass_baru or not pass_baru1:
        return {
            "status": False,
            "pesan": "Password gagal diganti, password tidak boleh kosong."
        }


    # 2. Validasi password baru
    pattern = (
        r"^(?=.*\d)"
        r"(?=.*[a-z])"
        r"(?=.*[A-Z])"
        r"(?=.*[^a-zA-Z0-9])"
        r"(?!.*\s)"
        r".{8,20}$"
    )

    if not re.match(pattern, pass_baru):
        return {
            "status": False,
            "pesan": (
                "Password baru harus 8-20 karakter "
                "dan memiliki angka, huruf besar, "
                "huruf kecil, dan simbol."
            )
        }


    # 3. Validasi password lama

    if pw_masih_md5:

        old_hash = hashlib.md5(
            pass_lama.encode()
        ).hexdigest()

        if old_hash != user.password:
            return {
                "status": False,
                "pesan": "Password lama tidak sesuai."
            }

    else:

        valid = bcrypt.checkpw(
            pass_lama.encode(),
            user.password.encode()
        )

        if not valid:
            return {
                "status": False,
                "pesan": "Password lama tidak sesuai."
            }


    # 4. Password baru tidak boleh sama
    if pass_baru == pass_lama:
        return {
            "status": False,
            "pesan": (
                "Password baru harus berbeda "
                "dengan password lama."
            )
        }


    # 5. Konfirmasi password
    if pass_baru != pass_baru1:
        return {
            "status": False,
            "pesan": (
                "Password baru dan konfirmasi "
                "tidak sama."
            )
        }


    # 6. Generate password baru bcrypt

    hashed_password = bcrypt.hashpw(
        pass_baru.encode(),
        bcrypt.gensalt()
    ).decode()


    # update object user
    # user.password = hashed_password


    return {
        "status": True,
        "pesan": "Password berhasil diganti.",
        "user": user
    }

In [ ]:
print(user.password)
print(len(user.password))

In [ ]:
print(user.id)
print(user.password)

In [ ]:
string(60) "$2b$12$FRq/0zbyElEwsRO..kejKuNsYLspY9m6E4Tm6dzFRzbpg.BRqU3VS" string(9) "123@Admin"

In [ ]:
"$2b$12$FRq/0zbyElEwsRO..kejKuNsYLspY9m6E4Tm6dzFRzbpg.BRqU3VS"

In [ ]:
import bcrypt


def check_password(password: str, hashed: str) -> bool:
    # ubah format bcrypt PHP menjadi format bcrypt Python
    if hashed.startswith("$2y$"):
        hashed = "$2b$" + hashed[4:]

    return bcrypt.checkpw(
        password.encode("utf-8"),
        hashed.encode("utf-8")
    )


print(
    check_password(
        "123@Admin",
        "$2y$10$l.833iKxH0adHbPArVETiexCZplxqIBZw4on29xopnLMVro17Bm.a"
    )
)

In [ ]:
from passlib.hash import bcrypt


# hashed = "$2y$10$l.833iKxH0adHbPArVETiexCZplxqIBZw4on29xopnLMVro17Bm.a"


print(
    bcrypt.verify(
        "123@Admin",
        hashed
    )
)

In [ ]:
password

In [ ]:
print(user.username)
print(user.password)
print(len(user.password))

In [ ]:
request = {
    "pass_lama": password,
    "pass_baru": password,
    "pass_baru1": password
}

In [ ]:
import bcrypt

print(
    bcrypt.checkpw(
        "123@Admin",
        hashed
    )
)

In [ ]:
from werkzeug.security import check_password_hash

check_password_hash(hashed, password)

In [ ]:
hash_pin(123456)
hash_password(password)

In [ ]:
from myapp.extensions import db

from myapp.features.auth.models import User
from myapp.features.auth.repository import (
    get_user_by_username,
    get_user_by_id,
)
from myapp.features.auth.services import authenticate

In [ ]:
from werkzeug.security import generate_password_hash

user = User(
    username="admin",
    email="admin@example.com",
    password_hash=generate_password_hash("123456"),
)

# db.session.add(user)
# db.session.commit()

# user

In [ ]:
user = get_user_by_username("admin")

user

In [ ]:
user = authenticate(
    username="admin",
    password="123456",
)

user

In [ ]:
user = get_user_by_username("admin")

user.email = "new@email.com"

db.session.commit()

user.email

In [ ]:
user = get_user_by_username("admin")

db.session.delete(user)
db.session.commit()

In [ ]:
from sqlalchemy import select

stmt = (
    select(User)
    .where(User.is_active == True)
    .order_by(User.username)
)

users = db.session.scalars(stmt).all()

users

In [ ]:
for user in users:
    print(user.id, user.username, user.email)